# 02. 외부 데이터 2차 EDA — 대회 클래스 불균형 보충 후보 수집

새로 다운로드해 압축을 푼 외부 데이터를 기준으로 **전체 이미지 현황 → 대회 bbox 분포 → 공통 클래스 보충 이미지 선별** 순서로 확인합니다.
전체 이미지 파일은 목록으로 집계하고, 픽셀은 예시만 읽습니다. 기존 누락 의심·OpenCV 분석은 제거했습니다.
원본 이미지·JSON·보정 기록·학습 데이터는 수정하지 않습니다.

- 기본 입력: `data/external/aihub_combo` 아래 모든 하위 데이터 폴더. 새 다운로드분도 자동 포함합니다.
- 다른 폴더를 쓰면 환경 변수 `AIHUB_EXTRACT_DIR`을 설정합니다. 다운로드·압축 해제는 자동 실행하지 않습니다.
- 대회 클래스는 `data/raw/train_annotations`의 라벨 기준입니다. test 정답 클래스는 알 수 없습니다.
- AI Hub의 공통 category_id=1 대신 `drug_N` 약 코드 숫자로 비교합니다. 이름·dl_idx는 클래스 키로 쓰지 않습니다.
- 기존 문서의 zip 제외 목록과 달리, 이번 EDA는 아래 `EXCLUDE=[]`로 현재 내려받은 전체 폴더를 봅니다.

앞 절의 157개 목표는 비교용이며, **최종 15절은 개수 상한 없이 외부 전용 약 0~1개인 이미지로 불균형을 줄입니다.** 대회에만 있는 클래스는 보충 대상에서 제외합니다. 외부 전용 클래스가 섞인 이미지를 라벨만 지워 사용하지 않습니다. 이 노트북은 후보 표를 만들며 원본 복사·학습 데이터 병합·YOLO 변환은 수행하지 않습니다.


In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists()),
    None,
)
if ROOT is None:
    raise RuntimeError("pill-bbox 저장소에서 실행하세요. Colab은 저장소를 먼저 준비하세요.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / ".env")
SEED = 42
EXCLUDE = []  # 현재 받은 전체 폴더를 확인한다. 필요할 때만 TL_2 등의 구분자를 넣는다.
extract_env = os.getenv("AIHUB_EXTRACT_DIR")
EXTRACT_DIR = Path(extract_env).expanduser() if extract_env else ROOT / "data/external/aihub_combo"
if not EXTRACT_DIR.is_absolute():
    EXTRACT_DIR = ROOT / EXTRACT_DIR
print("외부 데이터 폴더:", EXTRACT_DIR)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from data_process.aihub_data import list_combo_images, load_combo_annotations, zip_key
from data_process.data_processing import build_cat_df, build_img_df, list_images, load_annotations
from visualization import draw_bboxes, image_grid, set_korean_font, show_image

pd.set_option("display.max_columns", 40)
print("한글 폰트:", set_korean_font())

## 1. 전체 파일 현황

이미지·JSON 수를 폴더별로 집계합니다. 이미지 전체를 메모리에 읽지는 않습니다.
같은 파일 이름이 여러 폴더에 있으면 자동으로 하나를 선택하지 않고 중단해 중복 목록부터 확인합니다.


In [ ]:
if not EXTRACT_DIR.is_dir():
    raise FileNotFoundError(f"압축을 푼 외부 데이터 폴더가 없습니다: {EXTRACT_DIR}")
extracted_dirs = sorted(
    folder
    for folder in EXTRACT_DIR.iterdir()
    if folder.is_dir() and not folder.name.startswith(".") and zip_key(folder) not in EXCLUDE
)
if not extracted_dirs:
    raise FileNotFoundError("분석할 하위 폴더가 없습니다. EXTRACT_DIR과 EXCLUDE를 확인하세요.")
image_index = list_combo_images(extracted_dirs)
folder_inventory = pd.DataFrame(
    [
        {
            "폴더": folder.name,
            "이미지 수": int(image_index["zip_name"].eq(folder.name).sum()),
            "JSON 수": sum(1 for _ in folder.rglob("*.json")),
        }
        for folder in extracted_dirs
    ]
)
display(folder_inventory)
print(f"전체 이미지 파일: {len(image_index):,}장")
print(f"전체 JSON: {folder_inventory['JSON 수'].sum():,}개")
if image_index.empty:
    raise FileNotFoundError("이미지가 없습니다. TS/VS 원천 이미지 폴더를 확인하세요.")
duplicate_images = image_index[image_index["file_name"].duplicated(keep=False)]
print(f"동일 이름 중복 파일: {len(duplicate_images):,}개")
if not duplicate_images.empty:
    display(duplicate_images)
    raise ValueError("이미지 이름 중복을 먼저 확인해야 안전하게 라벨을 연결할 수 있습니다.")

## 2. 라벨 연결과 이미지 전체 요약

기존 로더로 전체 JSON 메타데이터를 읽습니다. 이미지가 없는 라벨도 버리지 않고 별도로 집계합니다.
`ann_df`는 모든 외부 라벨, `available_ann_df`는 실제 이미지 파일과 연결되는 라벨입니다.
클래스 비교에는 두 범위를 함께 표시하여 **라벨만 있고 이미지가 없는 클래스**를 구분합니다.


In [ ]:
ann_df, load_error_df = load_combo_annotations(extracted_dirs)
if ann_df.empty:
    display(load_error_df)
    raise ValueError("읽을 수 있는 외부 라벨이 없습니다. TL/VL 라벨 폴더를 확인하세요.")
invalid_class_rows = ann_df[ann_df["category_id"].isna()]
print(f"약 코드 해석 불가 bbox: {len(invalid_class_rows):,}개")
if not invalid_class_rows.empty:
    display(invalid_class_rows[["source_json", "drug_N", "dl_name"]].head(20))
    raise ValueError("해석할 수 없는 약 코드가 있어 클래스 비교를 중단합니다.")
image_names = set(image_index["file_name"])
missing_image_labels = ann_df[~ann_df["file_name"].isin(image_names)]
available_ann_df = ann_df[ann_df["file_name"].isin(image_names)].copy()
img_df = build_img_df(ann_df, list(image_index["path"])).merge(
    image_index[["file_name", "zip_name", "path"]], on="file_name", validate="one_to_one"
)
cat_df = build_cat_df(ann_df)
img_path = dict(zip(image_index["file_name"], image_index["path"], strict=True))
summary = pd.DataFrame(
    [
        ("디스크 전체 이미지", len(image_index)),
        ("라벨이 연결된 이미지", int(img_df["n_boxes"].gt(0).sum())),
        ("라벨 없는 이미지", int(img_df["n_boxes"].eq(0).sum())),
        ("전체 bbox 라벨", len(ann_df)),
        ("이미지와 연결된 bbox", len(available_ann_df)),
        ("라벨에만 있는 이미지 이름", missing_image_labels["file_name"].nunique()),
        ("전체 라벨 클래스", ann_df["category_id"].nunique()),
        ("이미지가 있는 클래스", available_ann_df["category_id"].nunique()),
        ("JSON 로드 오류·주의 기록", len(load_error_df)),
    ],
    columns=["항목", "개수"],
)
display(summary)
if not load_error_df.empty:
    display(load_error_df.head(20))
if not missing_image_labels.empty:
    print("이미지가 없는 라벨 예시 (자동 삭제하지 않음)")
    display(missing_image_labels[["file_name", "drug_N", "dl_name", "zip_name"]].head(20))

display(img_df.groupby("zip_name").agg(이미지=("file_name", "size"), bbox=("n_boxes", "sum")))
ax = img_df["zip_name"].value_counts().sort_index().plot.bar(figsize=(11, 4), rot=30)
ax.set(title="다운로드한 외부 이미지 전체 현황", xlabel="원천 폴더", ylabel="이미지 수")
plt.tight_layout()
plt.show()

## 3. 전체 범위의 이미지 예시

각 원천 폴더에서 최대 2장씩 seed=42로 뽑아 표시합니다. 원본 이미지 전체의 무결성 검사나 누락 판정은 아닙니다.
라벨이 있으면 기존 bbox와 약 이름을 함께 표시합니다.


In [ ]:
sample = img_df.groupby("zip_name", group_keys=False).sample(n=1, random_state=SEED)
# 각 폴더의 두 번째 예시가 가능하면 추가한다.
remaining = img_df.drop(sample.index)
if not remaining.empty:
    sample = pd.concat(
        [sample, remaining.groupby("zip_name", group_keys=False).sample(n=1, random_state=SEED)]
    ).sort_values(["zip_name", "file_name"])
fig, axes = image_grid(len(sample), cols=4)
for ax, row in zip(axes, sample.itertuples(), strict=True):
    show_image(
        ax, row.path, title=f"{row.zip_name} / bbox {row.n_boxes}개\n{row.file_name}", fontsize=6
    )
    draw_bboxes(ax, ann_df[ann_df["file_name"].eq(row.file_name)], label_col="dl_name")
fig.tight_layout()
plt.show()

## 4. 대회 train과 클래스 비교 — 공통 / 외부에만 / 대회에만

대회 원본 train 라벨의 `category_id`와 외부 라벨의 `drug_N` 숫자 변환 값을 비교합니다.
**외부에만 있는 클래스**는 대회 train 라벨에 없는 약 코드이며, 대회 test에도 없다는 뜻은 아닙니다.
전체 외부 라벨 기준 집합과 이미지가 실제 있는 집합을 모두 집계합니다. 이름 차이는 별도 참고값으로 표시하며 자동으로 합치거나 고치지 않습니다.


In [ ]:
raw_dir = ROOT / "data/raw"
if (raw_dir / "sprint_ai_project1_data").is_dir():
    raw_dir = raw_dir / "sprint_ai_project1_data"
if not (raw_dir / "train_annotations").is_dir():
    raise FileNotFoundError(f"대회 train_annotations가 없습니다: {raw_dir}")
comp_ann_df, comp_error_df = load_annotations(raw_dir / "train_annotations")
if comp_ann_df.empty or not comp_error_df.empty or comp_ann_df["category_id"].isna().any():
    display(comp_error_df)
    raise ValueError(
        "대회 라벨 로드 결과를 확인하세요. 불완전한 클래스 집합으로 비교하지 않습니다."
    )
comp_cat_df = build_cat_df(comp_ann_df)
comp_classes = set(comp_cat_df["category_id"].astype(int))
external_classes = set(cat_df["category_id"].astype(int))
available_classes = set(available_ann_df["category_id"].astype(int))
class_summary = pd.DataFrame(
    [
        {
            "비교 범위": scope,
            "대회 클래스": len(comp_classes),
            "외부 클래스": len(classes),
            "공통": len(classes & comp_classes),
            "외부에만": len(classes - comp_classes),
            "대회에만": len(comp_classes - classes),
        }
        for scope, classes in [
            ("전체 외부 라벨", external_classes),
            ("이미지가 있는 외부 라벨", available_classes),
        ]
    ]
)
display(class_summary)

available_counts = available_ann_df.groupby("category_id").agg(
    available_boxes=("annotation_id", "size"), available_images=("file_name", "nunique")
)
class_comparison = cat_df.merge(available_counts, on="category_id", how="left").merge(
    comp_cat_df[["category_id", "dl_name"]].rename(columns={"dl_name": "competition_name"}),
    on="category_id",
    how="left",
    validate="one_to_one",
)
class_comparison[["available_boxes", "available_images"]] = (
    class_comparison[["available_boxes", "available_images"]].fillna(0).astype(int)
)
class_comparison["구분"] = class_comparison["category_id"].map(
    lambda category: "공통" if category in comp_classes else "외부에만"
)
external_only_df = class_comparison[class_comparison["구분"].eq("외부에만")].copy()
shared_class_df = class_comparison[class_comparison["구분"].eq("공통")].copy()
competition_only_df = comp_cat_df[~comp_cat_df["category_id"].isin(external_classes)].copy()
print("대회 전용·외부 전용 클래스는 보충 대상에서 제외합니다.")
columns = ["category_id", "drug_N", "dl_name", "n_boxes", "n_images", "available_images"]
print("공통 클래스 (같은 약 코드의 이름 표기 비교)")
with pd.option_context("display.max_rows", None):
    display(shared_class_df[columns + ["competition_name"]].reset_index(drop=True))

## 5. 01_eda 기준 클래스 분포와 보충 목표

`01_eda.ipynb` 6.1과 같이 bbox 수로 분포를 봅니다. 원본 수와 파이프라인에서 생성한 **보정 v1** 수를 함께 표시하고 보충량은 v1 기준으로 계산합니다.
사용자 지정 기준: **대회 전체 최다 클래스의 bbox 수까지 공통 클래스만 보충**합니다. 대회 전용 클래스는 기준 수 계산에는 포함하되 추가 대상을 찾지 않습니다.

In [ ]:
import json

import numpy as np

from data_process.data_processing import bbox_flags, compare_name_codes
from data_process.make_labels import check_labels, file_sha256

label_path = ROOT / "data/processed/labels/labels_v1.csv"
label_meta_path = label_path.with_suffix(".json")
if not label_path.is_file() or not label_meta_path.is_file():
    raise FileNotFoundError(
        "보정 v1 라벨이 필요합니다. 기존 data_process.data_main --label v1을 먼저 실행하세요."
    )
label_meta = json.loads(label_meta_path.read_text())
if file_sha256(label_path) != label_meta["labels_sha256"]:
    raise ValueError("v1 라벨과 메타데이터 fingerprint가 다릅니다. 파이프라인을 다시 실행하세요.")
comp_corrected_df = pd.read_csv(label_path)
label_checks = check_labels(comp_corrected_df)
if any(
    label_checks[key] for key in ["bbox_missing", "out_of_image", "non_positive", "duplicate_label"]
):
    raise ValueError(f"보정 v1 라벨 검사를 통과하지 못했습니다: {label_checks}")
if set(comp_corrected_df["file_name"]) != set(comp_ann_df["file_name"]):
    raise ValueError("현재 대회 원본과 v1의 이미지 구성이 다릅니다. v1을 다시 생성하세요.")
raw_counts = comp_ann_df.groupby("category_id").size()
corrected_counts = comp_corrected_df.groupby("category_id").size()
common_classes = sorted(set(corrected_counts.index) & available_classes)
TARGET_BBOX = int(corrected_counts.max())
class_balance = (
    comp_cat_df[["category_id", "drug_N", "dl_name"]]
    .set_index("category_id")
    .assign(raw_bbox=raw_counts, competition_bbox=corrected_counts)
)
class_balance["common_class"] = class_balance.index.isin(common_classes)
class_balance["target_bbox"] = TARGET_BBOX
class_balance["needed_bbox"] = (TARGET_BBOX - class_balance["competition_bbox"]).clip(lower=0)
class_balance.loc[~class_balance["common_class"], "needed_bbox"] = 0
class_balance["group"] = np.select(
    [class_balance["competition_bbox"].le(5), class_balance["competition_bbox"].le(20)],
    ["희귀 (≤5)", "소수 (6–20)"],
    default="다수 (≥21)",
)
print(f"보정 v1: {len(comp_corrected_df):,} bbox / {len(corrected_counts)}종")
print(f"공통 {len(common_classes)}종의 목표: 클래스당 {TARGET_BBOX:,} bbox")
with pd.option_context("display.max_rows", None):
    display(class_balance.sort_values("competition_bbox"))
plot_counts = class_balance.sort_values("competition_bbox")
ax = plot_counts["competition_bbox"].plot.barh(figsize=(11, 14))
ax.axvline(TARGET_BBOX, color="red", linestyle="--", label=f"보충 목표 {TARGET_BBOX}")
ax.set(title="대회 보정 v1 클래스별 bbox 분포", xlabel="bbox 수", ylabel="약 코드")
ax.legend()
plt.tight_layout()
plt.show()

## 6. 공통 알약 후보 수집과 이미지 단위 검사

`common_bbox_candidates`는 공통 클래스의 개별 bbox 후보 전체입니다. 여기에 외부 전용 약이 함께 찍힌 이미지도 있을 수 있습니다.
실제 원본 이미지 보충용 `candidate_annotations`는 **파일명 코드와 라벨이 일치하고, 모든 클래스가 공통이며, 좌표·중복 문제가 없는 이미지**의 모든 라벨입니다.
대회 train/test와 동일 이름 또는 동일 combo의 이미지는 제외합니다. 내용 해시·닮은 촬영 세팅까지 검증한 누수 검사는 아니므로 후보가 곧 최종 학습 승인은 아닙니다.

In [ ]:
common_bbox_candidates = available_ann_df[
    available_ann_df["category_id"].isin(common_classes)
].merge(image_index[["file_name", "path"]], on="file_name", validate="many_to_one")

name_check = compare_name_codes(img_df, available_ann_df).set_index("file_name")
box_flags = bbox_flags(available_ann_df)
numeric = available_ann_df[["x", "y", "w", "h", "width", "height"]].apply(
    pd.to_numeric, errors="coerce"
)
invalid_box = (
    ~np.isfinite(numeric).all(axis=1)
    | numeric[["width", "height"]].le(0).any(axis=1)
    | box_flags["out_of_image"]
    | box_flags["non_positive"]
    | available_ann_df.duplicated(["file_name", "category_id"], keep=False)
)
invalid_files = set(available_ann_df.loc[invalid_box, "file_name"])
classes_by_image = available_ann_df.groupby("file_name")["category_id"].agg(set)
comp_image_names = set()
for split in ["train", "test"]:
    image_dir = raw_dir / f"{split}_images"
    if not image_dir.is_dir():
        raise FileNotFoundError(f"대회 {split} 이미지 폴더가 필요합니다: {image_dir}")
    comp_image_names.update(path.name for path in list_images(image_dir))
comp_combos = set(comp_corrected_df["file_name"].str.split("_").str[0])
comp_combos.update(name.split("_")[0] for name in comp_image_names if name.startswith("K-"))

candidate_image_review = img_df[["file_name", "combo", "path"]].copy()
candidate_image_review["has_common_class"] = candidate_image_review["file_name"].map(
    lambda name: bool(classes_by_image.get(name, set()) & set(common_classes))
)
candidate_image_review["only_common_classes"] = candidate_image_review["file_name"].map(
    lambda name: (
        bool(classes_by_image.get(name, set())) and classes_by_image[name].issubset(common_classes)
    )
)
candidate_image_review["complete_codes"] = candidate_image_review["file_name"].map(
    name_check["missing_codes"].map(len).eq(0) & name_check["extra_codes"].map(len).eq(0)
)
candidate_image_review["valid_boxes"] = ~candidate_image_review["file_name"].isin(invalid_files)
candidate_image_review["competition_overlap"] = candidate_image_review["file_name"].isin(
    comp_image_names
) | candidate_image_review["combo"].isin(comp_combos)
candidate_image_review["eligible"] = (
    candidate_image_review["only_common_classes"]
    & candidate_image_review["complete_codes"]
    & candidate_image_review["valid_boxes"]
    & ~candidate_image_review["competition_overlap"]
)
candidate_images = candidate_image_review[candidate_image_review["eligible"]].copy()
candidate_annotations = available_ann_df[
    available_ann_df["file_name"].isin(candidate_images["file_name"])
].merge(image_index[["file_name", "path"]], on="file_name", validate="many_to_one")
class_balance["external_all_common_bbox"] = (
    common_bbox_candidates.groupby("category_id").size().reindex(class_balance.index, fill_value=0)
)
class_balance["eligible_external_bbox"] = (
    candidate_annotations.groupby("category_id").size().reindex(class_balance.index, fill_value=0)
)
class_balance["pool_shortfall"] = (
    class_balance["needed_bbox"] - class_balance["eligible_external_bbox"]
).clip(lower=0)
print(f"공통 클래스 개별 bbox 후보: {len(common_bbox_candidates):,}개")
print(
    f"전체 이미지 보충 가능 후보: {len(candidate_images):,}장 / {len(candidate_annotations):,} bbox"
)
display(
    candidate_image_review[
        [
            "has_common_class",
            "only_common_classes",
            "complete_codes",
            "valid_boxes",
            "competition_overlap",
            "eligible",
        ]
    ]
    .sum()
    .rename("이미지 수")
)
with pd.option_context("display.max_rows", None):
    display(class_balance[class_balance["common_class"]].sort_values("competition_bbox"))

## 7. 공통 알약을 클래스별 목표 수까지 수집 — 개별 bbox 후보

사용할 알약은 **대회에 있는 공통 클래스만** 모읍니다. 외부 전용 알약은 선택하지 않습니다.
조합 이미지 안에서 해당 클래스의 bbox만 후보로 수집하므로, 원본 이미지를 그대로 쓰면서 다른 클래스의 라벨만 지우는 방식으로 학습하면 안 됩니다.

- `selected_pills`: 대회 bbox 수를 157개(현재 최다 클래스)까지 채우기 위해 선택한 알약의 이미지 경로·좌표·클래스·출처 JSON
- `pill_balance_report`: 클래스별 현재 수·추가 후보 수·예상 합계·부족량
- `pill_candidate_pool`: 공통 약 전체 bbox 후보 중 기본 좌표 검사를 통과하고 대회와 동일 파일명이 아닌 후보

이것은 향후 crop/Copy-Paste용 **선별 목록**이며, 잘라낸 픽셀이나 학습 라벨을 생성하지 않습니다. 함께 찍힌 약의 crop 침범 여부는 이미지 검수가 필요합니다.
대회와 동일 combo인 후보는 `shares_competition_combo=True`로 표시합니다. 이는 선택 풀에 포함하되, 향후 validation에 같은 combo/세팅이 들어가는 fold에서는 학습 후보에서 제외하고 재선별해야 합니다.

In [ ]:
pill_candidate_pool = common_bbox_candidates[
    ~common_bbox_candidates["file_name"].isin(invalid_files | comp_image_names)
].copy()
pill_candidate_pool["combo"] = pill_candidate_pool["file_name"].str.split("_").str[0]
pill_candidate_pool["shares_competition_combo"] = pill_candidate_pool["combo"].isin(comp_combos)
pill_candidate_pool = pill_candidate_pool.drop_duplicates(
    ["file_name", "category_id", "x", "y", "w", "h"]
)
# 같은 클래스의 서로 다른 이미지를 seed를 고정하여 비복원 추출한다.
shuffled_pills = pill_candidate_pool.sample(frac=1, random_state=SEED)
pill_order = shuffled_pills.groupby("category_id").cumcount()
pill_need = shuffled_pills["category_id"].map(class_balance["needed_bbox"])
selected_pills = (
    shuffled_pills[pill_order.lt(pill_need)]
    .sort_values(["category_id", "file_name"])
    .reset_index(drop=True)
)
pill_balance_report = class_balance.loc[common_classes].copy()
pill_balance_report["candidate_bbox"] = (
    pill_candidate_pool.groupby("category_id").size().reindex(common_classes, fill_value=0)
)
pill_balance_report["selected_bbox"] = (
    selected_pills.groupby("category_id").size().reindex(common_classes, fill_value=0)
)
pill_balance_report["after_bbox"] = (
    pill_balance_report["competition_bbox"] + pill_balance_report["selected_bbox"]
)
pill_balance_report["remaining_bbox"] = (TARGET_BBOX - pill_balance_report["after_bbox"]).clip(
    lower=0
)
pill_balance_report["status"] = np.where(
    pill_balance_report["remaining_bbox"].eq(0), "목표 충족", "후보 부족"
)
assert set(selected_pills["category_id"]).issubset(common_classes)
assert not selected_pills.duplicated(["file_name", "category_id"]).any()
assert pill_balance_report["after_bbox"].le(TARGET_BBOX).all()
n_pill_images = selected_pills["file_name"].nunique()
print(f"수집한 공통 알약: {len(selected_pills):,} bbox / {n_pill_images:,}개 이미지")
n_pill_met = int(pill_balance_report["remaining_bbox"].eq(0).sum())
print(f"목표 {TARGET_BBOX}개 충족: {n_pill_met}/{len(common_classes)}종")
with pd.option_context("display.max_rows", None):
    display(
        pill_balance_report[
            [
                "dl_name",
                "competition_bbox",
                "needed_bbox",
                "candidate_bbox",
                "selected_bbox",
                "after_bbox",
                "remaining_bbox",
                "status",
            ]
        ]
    )
display(
    selected_pills[
        [
            "category_id",
            "dl_name",
            "file_name",
            "path",
            "x",
            "y",
            "w",
            "h",
            "source_json",
            "shares_competition_combo",
        ]
    ].head(20)
)
pill_plot = pill_balance_report.sort_values("competition_bbox")
ax = pill_plot[["competition_bbox", "selected_bbox"]].plot.barh(
    stacked=True, figsize=(12, 14), color=["steelblue", "darkorange"]
)
ax.axvline(TARGET_BBOX, color="red", linestyle="--", label=f"목표 {TARGET_BBOX}")
ax.set(title="공통 알약 클래스별 보충 후보 수집 결과", xlabel="bbox 수", ylabel="약 코드")
ax.legend()
plt.tight_layout()
plt.show()

### 7.1 클래스별 외부 데이터 반입 수량 표

현재 데이터와 선별 결과로 수량을 자동 계산합니다. 공통 클래스만 표시하고 마지막에 합계를 붙입니다.


In [ ]:
# 앞 셀의 실제 선별 결과로 클래스별 반입 수량 표를 만든다.
if "pill_balance_report" not in globals():
    raise RuntimeError("먼저 7절의 공통 알약 후보 선별 셀을 실행하세요.")

supplement_counts = (
    pill_balance_report[
        ["dl_name", "competition_bbox", "needed_bbox", "selected_bbox", "remaining_bbox"]
    ]
    .rename(
        columns={
            "dl_name": "클래스",
            "competition_bbox": "대회 현재",
            "needed_bbox": "추가 필요",
            "selected_bbox": "외부 선별 가능",
            "remaining_bbox": "남은 부족",
        }
    )
    .reset_index(drop=True)
)
count_columns = ["대회 현재", "추가 필요", "외부 선별 가능", "남은 부족"]
supplement_counts[count_columns] = supplement_counts[count_columns].astype(int)
supplement_counts.loc[len(supplement_counts)] = {
    "클래스": "합계",
    **supplement_counts[count_columns].sum().to_dict(),
}
print(f"공통 클래스별 목표: {TARGET_BBOX} bbox / 단위: 알약 bbox 개수 (이미지 장수 아님)")
print("외부 선별 가능은 개별 알약 후보 수이며, 실제 crop·증강·학습 데이터 병합 전입니다.")
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(supplement_counts.style.format({column: "{:,.0f}" for column in count_columns}))

## 8. 원본 이미지 전체를 추가할 경우의 별도 선별

조합 이미지를 하나 고르면 함께 있는 공통 약들의 bbox도 모두 추가합니다. 부족한 클래스의 해소량을 우선하는 탐욕 선택으로, 어느 클래스도 목표 수를 넘지 않는 이미지만 고릅니다. seed=42로 같은 입력의 동률 선택을 재현합니다.
조합 구성·후보 수 제약 때문에 정확한 균형을 보장하는 최적화는 아닙니다. 남은 부족량은 그대로 보고하며 이미지를 중복 복제해 채우지 않습니다.

- `selected_images`: 선택한 원본 이미지 경로와 combo
- `selected_annotations`: 선택 이미지의 모든 bbox·약 코드·출처 JSON
- `balance_report`: 대회 수 / 선택 추가 수 / 추가 후 수 / 남은 부족량

원본 파일 이동·복사와 train/val 병합은 하지 않습니다. 이후 학습 연동 시에는 fold별 train 기준으로 다시 선별하고 외부 combo·촬영 세팅과 validation의 중복을 확인해야 합니다.

In [ ]:
image_class_counts = pd.crosstab(
    candidate_annotations["file_name"], candidate_annotations["category_id"]
).reindex(columns=common_classes, fill_value=0)
# 행 순서를 seed로 섞은 후 동률은 첫 번째를 골라 결과를 재현한다.
image_class_counts = image_class_counts.sample(frac=1, random_state=SEED)
counts_matrix = image_class_counts.to_numpy(dtype=int)
remaining = class_balance.loc[common_classes, "needed_bbox"].to_numpy(dtype=int, copy=True)
active = np.ones(len(image_class_counts), dtype=bool)
selected_positions = []
while active.any() and remaining.any():
    fits = active & (counts_matrix <= remaining).all(axis=1)
    if not fits.any():
        break
    # 부족량이 많이 남은 클래스의 bbox를 우선하되 함께 추가되는 모든 bbox를 계산한다.
    weights = remaining / np.maximum(
        class_balance.loc[common_classes, "needed_bbox"].to_numpy(dtype=int), 1
    )
    scores = counts_matrix @ weights
    scores[~fits] = -1
    position = int(scores.argmax())
    if scores[position] <= 0:
        break
    selected_positions.append(position)
    remaining -= counts_matrix[position]
    active[position] = False

selected_names = set(image_class_counts.index[selected_positions])
selected_images = candidate_images[candidate_images["file_name"].isin(selected_names)].copy()
selected_annotations = candidate_annotations[
    candidate_annotations["file_name"].isin(selected_names)
].copy()
balance_report = class_balance.copy()
balance_report["added_bbox"] = (
    selected_annotations.groupby("category_id").size().reindex(balance_report.index, fill_value=0)
)
balance_report["after_bbox"] = balance_report["competition_bbox"] + balance_report["added_bbox"]
balance_report["remaining_bbox"] = np.where(
    balance_report["common_class"],
    (TARGET_BBOX - balance_report["after_bbox"]).clip(lower=0),
    0,
)
balance_report["status"] = np.select(
    [~balance_report["common_class"], balance_report["remaining_bbox"].eq(0)],
    ["대회 전용: 보충 제외", "목표 충족"],
    default="미충족: 후보·조합 제약",
)
assert set(selected_annotations["category_id"]).issubset(common_classes)
assert not selected_images["file_name"].duplicated().any()
assert balance_report.loc[common_classes, "after_bbox"].le(TARGET_BBOX).all()
assert balance_report.loc[~balance_report["common_class"], "added_bbox"].eq(0).all()
print(f"선택: 이미지 {len(selected_images):,}장 / bbox {len(selected_annotations):,}개")
n_met = int(balance_report.loc[common_classes, "remaining_bbox"].eq(0).sum())
print(f"공통 {len(common_classes)}종 중 목표 충족: {n_met}종")
with pd.option_context("display.max_rows", None):
    display(
        balance_report.sort_values(["common_class", "competition_bbox"], ascending=[False, True])
    )
ordered = balance_report.loc[common_classes].sort_values("competition_bbox")
ax = ordered[["competition_bbox", "added_bbox"]].plot.barh(
    stacked=True, figsize=(12, 14), color=["steelblue", "darkorange"]
)
ax.axvline(TARGET_BBOX, color="red", linestyle="--", label=f"목표 {TARGET_BBOX}")
ax.set(
    title="공통 클래스의 외부 데이터 보충 전·후 (후보 선별 결과)",
    xlabel="bbox 수",
    ylabel="약 코드",
)
ax.legend()
plt.tight_layout()
plt.show()

## 9. 원본 이미지 보충 후보 시각 확인

선택 이미지 최대 8장을 표시합니다. 같은 이미지의 모든 라벨을 그립니다. 이 표들은 메모리의 후보이며 학습 입력은 기존 파이프라인으로 별도 만들어야 합니다.

In [ ]:
display(selected_images.head(20))
display(
    selected_annotations[
        ["file_name", "category_id", "dl_name", "x", "y", "w", "h", "source_json", "path"]
    ].head(20)
)
if not selected_images.empty:
    examples = selected_images.sample(min(8, len(selected_images)), random_state=SEED)
    fig, axes = image_grid(len(examples), cols=4)
    for ax, row in zip(axes, examples.itertuples(), strict=True):
        show_image(ax, row.path, title=row.file_name, fontsize=6)
        draw_bboxes(
            ax,
            selected_annotations[selected_annotations["file_name"].eq(row.file_name)],
            label_col="dl_name",
        )
    fig.tight_layout()
    plt.show()
else:
    print(
        "조건에 맞는 보충 이미지가 없습니다. candidate_image_review와 pool_shortfall을 확인하세요."
    )

## 10. 추가 예정 알약의 bbox 유무 — 이미지별 점검

1. **이미지의 알약 수**: 파일명에 적힌 약 코드 수입니다. 실제 픽셀을 보고 센 개수는 아니므로 눈으로 검수해야 합니다.
2. **이미지별 bbox 수**: 해당 이미지에 연결된 **전체 외부 라벨** 수입니다. 선택한 알약만 세지 않습니다.
3. **필요한 알약의 bbox 유무**: 파일명의 필요한 약 코드와 실제 라벨을 클래스별로 대조합니다. bbox가 있어도 좌표 오류·중복이면 따로 표시합니다.

선택된 알약만 검사하면 bbox가 없는 약을 놓칠 수 있어, **보충 대상 약 코드가 파일명에 있는 전체 외부 이미지**를 검사합니다. `추가 예정 이미지`·`선택한 알약` 열로 7절에서 실제 선택한 후보를 구분합니다. 이름 기준 후보와 선택 라벨이 불일치하는 경우도 남깁니다.

- `image_bbox_audit`: 이미지별 알약 수·전체 bbox 수·필요한 약의 bbox 없는 코드
- `pill_bbox_audit`: 이미지 × 필요한 클래스별 bbox 존재·좌표 검사·선택 여부
- `class_bbox_audit`: 클래스별 bbox 있는 이미지·없는 이미지·선택한 이미지 수

전체 표는 변수에 보관하고 화면에는 클래스 요약과 이미지별 앞 50행, 문제 사례 앞 50행을 표시합니다. 표시는 `BBOX_AUDIT_PREVIEW_ROWS`로 조정합니다. 이 검사는 파일명·라벨 기준이며 알약 종류와 박스의 시각적 정확성까지 보증하지 않습니다.


In [ ]:
BBOX_AUDIT_PREVIEW_ROWS = 50
required_variables = ["selected_pills", "class_balance", "img_df", "available_ann_df"]
if any(name not in globals() for name in required_variables):
    raise RuntimeError("먼저 7절의 공통 알약 후보 선별까지 실행하세요.")

needed_classes = set(
    class_balance.index[class_balance["common_class"] & class_balance["needed_bbox"].gt(0)]
)
# 파일명에 있지만 라벨이 없는 약도 검사 대상에 포함한다.
audit_images = img_df[["file_name", "combo", "path", "n_codes_in_name"]].copy()
audit_images["필요한 코드"] = audit_images["combo"].map(
    lambda combo: sorted({int(code) for code in combo.split("-")[1:]} & needed_classes)
)
expected_pairs = (
    audit_images[["file_name", "필요한 코드"]]
    .explode("필요한 코드")
    .dropna(subset=["필요한 코드"])
    .rename(columns={"필요한 코드": "category_id"})
)
expected_pairs["category_id"] = expected_pairs["category_id"].astype(int)
expected_pairs["파일명에 있는 약"] = True
selected_pairs = selected_pills[["file_name", "category_id"]].drop_duplicates().copy()
selected_pairs["선택한 알약"] = True
pill_bbox_audit = expected_pairs.merge(
    selected_pairs, on=["file_name", "category_id"], how="outer", validate="one_to_one"
)
pill_bbox_audit[["파일명에 있는 약", "선택한 알약"]] = (
    pill_bbox_audit[["파일명에 있는 약", "선택한 알약"]].fillna(False).astype(bool)
)

# bbox 존재와 사용 가능한 좌표를 구분한다. 다른 알약의 bbox로 대체하지 않는다.
audit_numeric = available_ann_df[["x", "y", "w", "h", "width", "height"]].apply(
    pd.to_numeric, errors="coerce"
)
audit_flags = bbox_flags(available_ann_df)
audit_box_valid = (
    np.isfinite(audit_numeric).all(axis=1)
    & audit_numeric[["width", "height"]].gt(0).all(axis=1)
    & ~audit_flags["out_of_image"]
    & ~audit_flags["non_positive"]
    & ~available_ann_df.duplicated(["file_name", "category_id"], keep=False)
)
boxes_by_pill = (
    available_ann_df.assign(valid_bbox=audit_box_valid)
    .groupby(["file_name", "category_id"])
    .agg(n_bbox=("category_id", "size"), n_valid_bbox=("valid_bbox", "sum"))
    .rename(columns={"n_bbox": "bbox 수", "n_valid_bbox": "유효 bbox 수"})
    .reset_index()
)
pill_bbox_audit = pill_bbox_audit.merge(
    boxes_by_pill, on=["file_name", "category_id"], how="left", validate="one_to_one"
)
for column in ["bbox 수", "유효 bbox 수"]:
    pill_bbox_audit[column] = pill_bbox_audit[column].fillna(0).astype(int)
pill_bbox_audit["알약 이름"] = pill_bbox_audit["category_id"].map(class_balance["dl_name"])
pill_bbox_audit["bbox 있음"] = pill_bbox_audit["bbox 수"].gt(0)
pill_bbox_audit["bbox 상태"] = np.select(
    [~pill_bbox_audit["bbox 있음"], pill_bbox_audit["유효 bbox 수"].eq(0)],
    ["없음", "있음: 좌표·중복 확인 필요"],
    default="있음: 기본 좌표 검사 통과",
)
pill_bbox_audit = pill_bbox_audit.sort_values(["file_name", "category_id"]).reset_index(drop=True)

# 선택되지 않은 이미지도 포함하되 필요한 약이 전혀 없는 이미지는 제외한다.
image_bbox_audit = audit_images[audit_images["file_name"].isin(pill_bbox_audit["file_name"])].copy()
all_bbox_counts = available_ann_df.groupby("file_name").size()
image_bbox_audit = image_bbox_audit.rename(columns={"n_codes_in_name": "알약 수 (파일명 기준)"})
image_bbox_audit["전체 bbox 수"] = (
    image_bbox_audit["file_name"].map(all_bbox_counts).fillna(0).astype(int)
)
image_bbox_audit["필요한 약 수"] = image_bbox_audit["필요한 코드"].map(len)
missing_by_image = (
    pill_bbox_audit[~pill_bbox_audit["bbox 있음"]].groupby("file_name")["category_id"].agg(list)
)
invalid_by_image = (
    pill_bbox_audit[pill_bbox_audit["bbox 있음"] & pill_bbox_audit["유효 bbox 수"].eq(0)]
    .groupby("file_name")["category_id"]
    .agg(list)
)
image_bbox_audit["bbox 없는 필요 코드"] = image_bbox_audit["file_name"].map(
    lambda name: missing_by_image.get(name, [])
)
image_bbox_audit["좌표·중복 확인 코드"] = image_bbox_audit["file_name"].map(
    lambda name: invalid_by_image.get(name, [])
)
image_bbox_audit["필요한 약 모두 bbox 있음"] = (
    image_bbox_audit["bbox 없는 필요 코드"].map(len).eq(0)
)
image_bbox_audit["알약 수와 bbox 수 일치"] = (
    image_bbox_audit["알약 수 (파일명 기준)"] == image_bbox_audit["전체 bbox 수"]
)
image_bbox_audit["추가 예정 이미지"] = image_bbox_audit["file_name"].isin(
    selected_pills["file_name"]
)
image_bbox_audit = image_bbox_audit.sort_values(
    ["추가 예정 이미지", "필요한 약 모두 bbox 있음", "file_name"],
    ascending=[False, True, True],
).reset_index(drop=True)

class_bbox_audit = (
    pill_bbox_audit.assign(
        missing_bbox=~pill_bbox_audit["bbox 있음"],
        invalid_bbox=pill_bbox_audit["bbox 있음"] & pill_bbox_audit["유효 bbox 수"].eq(0),
    )
    .groupby("category_id")
    .agg(
        검사_이미지=("file_name", "nunique"),
        bbox_있는_이미지=("bbox 있음", "sum"),
        bbox_없는_이미지=("missing_bbox", "sum"),
        좌표_중복_확인_이미지=("invalid_bbox", "sum"),
        선택한_이미지=("선택한 알약", "sum"),
    )
    .reindex(sorted(needed_classes), fill_value=0)
)
class_bbox_audit.insert(0, "알약 이름", class_bbox_audit.index.map(class_balance["dl_name"]))
image_problems = image_bbox_audit[
    ~image_bbox_audit["필요한 약 모두 bbox 있음"]
    | ~image_bbox_audit["알약 수와 bbox 수 일치"]
    | image_bbox_audit["좌표·중복 확인 코드"].map(len).gt(0)
]
pill_problems = pill_bbox_audit[
    ~pill_bbox_audit["bbox 있음"]
    | pill_bbox_audit["유효 bbox 수"].eq(0)
    | ~pill_bbox_audit["파일명에 있는 약"]
]
print(f"검사 이미지 {len(image_bbox_audit):,}장 / 점검 대상 이미지 {len(image_problems):,}장")
selected_check = pill_bbox_audit[pill_bbox_audit["선택한 알약"]]
print(
    f"선택한 알약 {len(selected_check):,}개 중 bbox 없음 "
    f"{(~selected_check['bbox 있음']).sum():,}개 / 유효 좌표 없음 "
    f"{selected_check['유효 bbox 수'].eq(0).sum():,}개"
)
print("클래스별 필요한 알약의 bbox 유무")
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(class_bbox_audit)
print("이미지별 알약 수·전체 bbox 수·필요한 약 bbox 유무")
display(image_bbox_audit.head(BBOX_AUDIT_PREVIEW_ROWS))
print("이미지별 점검 대상 (개수 일치만으로 라벨 완전성을 확정하지 않음)")
display(image_problems.head(BBOX_AUDIT_PREVIEW_ROWS))
print("필요한 알약별 bbox 누락·좌표 문제·파일명 불일치")
display(pill_problems.head(BBOX_AUDIT_PREVIEW_ROWS))

## 11. 한눈에 보기 — 클래스 번호·개수와 bbox 점검

위의 7절 후보 선별과 10절 bbox 검사 후 실행합니다.

- **클래스 번호 / 개수**: 외부에서 추가할 개별 알약 bbox 후보 수입니다. 대회 데이터와 합친 수가 아닙니다.
- **이미지별 알약 수 / bbox 수**: 선택한 알약이 들어 있는 원본 이미지의 전체 약 코드 수·전체 bbox 수입니다. 알약 수는 파일명 기준이며 육안으로 센 실제 수가 아닙니다.
- **클래스별 bbox 있음 / 없음**: 해당 범위의 이미지에 필요한 약 코드가 있지만 라벨이 있는지 없는지 대조합니다. 같은 이미지에 함께 있는 다른 필요한 약도 포함합니다.

기본은 **추가 예정 이미지**만 표시합니다. `SUMMARY_SELECTED_ONLY=False`로 바꾸면 10절에서 검사한 전체 외부 후보 범위를 봅니다.
긴 파일명 대신 이미지 번호로 표시합니다. 원본 파일명·경로는 `image_number_lookup`에서 같은 번호로 찾을 수 있습니다.
화면에는 이미지별 앞 50행만 표시하며 전체 결과는 `simple_image_counts`에 있습니다. `SUMMARY_IMAGE_ROWS`로 표시 행 수를 조절합니다.


In [ ]:
SUMMARY_SELECTED_ONLY = True
SUMMARY_IMAGE_ROWS = 50
if any(
    name not in globals()
    for name in ["selected_pills", "image_bbox_audit", "pill_bbox_audit", "pill_balance_report"]
):
    raise RuntimeError("먼저 7절 후보 선별과 10절 bbox 점검 셀을 실행하세요.")

# 클래스 번호와 실제 추가할 개수만 표시한다.
simple_class_counts = (
    selected_pills.groupby("category_id")
    .size()
    .reindex(pill_balance_report.index, fill_value=0)
    .rename_axis("클래스 번호")
    .rename("개수")
    .reset_index()
)
print(
    f"추가할 알약 후보: {len(simple_class_counts)}종 / {simple_class_counts['개수'].sum():,} bbox"
)
with pd.option_context("display.max_rows", None):
    display(simple_class_counts.style.hide(axis="index"))

fig, axes = plt.subplots(1, 2, figsize=(16, 10), sharex=True)
for ax, positions in zip(axes, np.array_split(np.arange(len(simple_class_counts)), 2), strict=True):
    part = simple_class_counts.iloc[positions]
    ax.barh(part["클래스 번호"].astype(str), part["개수"], color="steelblue")
    ax.invert_yaxis()
    ax.set(xlabel="추가 bbox 개수", ylabel="클래스 번호")
    ax.bar_label(ax.containers[0], padding=3, fontsize=8)
    ax.set_xlim(0, max(1, simple_class_counts["개수"].max()) * 1.18)
fig.suptitle("클래스별 추가할 알약 bbox 개수")
fig.tight_layout()
plt.show()

scope_images = image_bbox_audit.copy()
if SUMMARY_SELECTED_ONLY:
    scope_images = scope_images[scope_images["추가 예정 이미지"]].copy()
scope_images = scope_images.sort_values("file_name").reset_index(drop=True)
scope_images["이미지 번호"] = np.arange(1, len(scope_images) + 1)
image_number_lookup = scope_images[["이미지 번호", "file_name", "path"]].copy()
simple_image_counts = scope_images[["이미지 번호", "알약 수 (파일명 기준)", "전체 bbox 수"]].rename(
    columns={"알약 수 (파일명 기준)": "알약 수", "전체 bbox 수": "bbox 수"}
)
simple_image_counts["필요 약 bbox 없음"] = scope_images["bbox 없는 필요 코드"].map(len)
print("검사 범위:", "추가 예정 이미지" if SUMMARY_SELECTED_ONLY else "전체 외부 후보 이미지")
print(f"이미지 {len(simple_image_counts):,}장 / 알약 수는 파일명 기준")
display(simple_image_counts.head(SUMMARY_IMAGE_ROWS).style.hide(axis="index"))

pill_count_distribution = (
    simple_image_counts["알약 수"]
    .value_counts()
    .sort_index()
    .rename_axis("알약 수")
    .rename("이미지 수")
    .reset_index()
)
bbox_count_distribution = (
    simple_image_counts["bbox 수"]
    .value_counts()
    .sort_index()
    .rename_axis("bbox 수")
    .rename("이미지 수")
    .reset_index()
)
print("이미지에 알약이 몇 개씩 있는지")
display(pill_count_distribution.style.hide(axis="index"))
print("이미지에 bbox가 몇 개씩 있는지")
display(bbox_count_distribution.style.hide(axis="index"))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, distribution, key in zip(
    axes, [pill_count_distribution, bbox_count_distribution], ["알약 수", "bbox 수"], strict=True
):
    ax.bar(distribution[key].astype(str), distribution["이미지 수"], color="steelblue")
    ax.set(xlabel=key, ylabel="이미지 수", title=f"이미지별 {key} 분포")
    ax.bar_label(ax.containers[0], padding=3)
    ax.margins(y=0.18)
fig.tight_layout()
plt.show()

# 필요한 약마다 이미지에 해당 클래스의 bbox가 있는지 계산한다.
scope_pills = pill_bbox_audit[pill_bbox_audit["file_name"].isin(scope_images["file_name"])].copy()
simple_bbox_presence = (
    scope_pills.assign(missing_bbox=~scope_pills["bbox 있음"])
    .groupby("category_id")
    .agg(있음=("bbox 있음", "sum"), 없음=("missing_bbox", "sum"))
    .reindex(pill_balance_report.index, fill_value=0)
    .rename_axis("클래스 번호")
    .reset_index()
)
print("필요한 클래스의 bbox 있음 / 없음 (단위: 이미지 수)")
with pd.option_context("display.max_rows", None):
    display(simple_bbox_presence.style.hide(axis="index"))
print("있음=0, 없음=0은 해당 검사 범위에 그 클래스의 검사 대상 이미지가 없다는 뜻입니다.")
fig, axes = plt.subplots(1, 2, figsize=(16, 10), sharex=True)
for ax, positions in zip(
    axes, np.array_split(np.arange(len(simple_bbox_presence)), 2), strict=True
):
    part = simple_bbox_presence.iloc[positions]
    labels = part["클래스 번호"].astype(str)
    ax.barh(labels, part["있음"], label="bbox 있음", color="seagreen")
    ax.barh(labels, part["없음"], left=part["있음"], label="bbox 없음", color="tomato")
    ax.invert_yaxis()
    ax.set(xlabel="이미지 수", ylabel="클래스 번호")
    ax.legend()
fig.suptitle("필요한 알약의 클래스별 bbox 유무")
fig.tight_layout()
plt.show()

## 12. 추가 예정 이미지 — bbox 누락·겹침만 확인

검사 범위는 `selected_pills`의 알약이 들어 있는 **추가 예정 원본 이미지**입니다. 누락은 파일명에 있는 **모든 약 코드**와 전체 라벨을 대조합니다. 추가할 공통 클래스뿐 아니라 같은 이미지에 함께 찍힌 외부 전용 클래스도 포함합니다. bbox 라벨이 없거나 bbox 좌표가 비어 있는 경우를 구분합니다.

겹침은 같은 이미지의 **모든 클래스 bbox**를 비교하여 **IoU > 0**인 쌍만 남깁니다. 테두리만 닿은 박스는 제외합니다. `선택 bbox 포함`으로 추가 예정 알약이 해당 겹침에 포함되는지 구분합니다. 겹침 자체가 라벨 오류라는 뜻은 아닙니다.

- `missing_bbox_cases`: 이미지 번호·클래스 번호·누락 유형
- `overlap_bbox_cases`: 이미지 번호·두 클래스 번호·IoU·선택 bbox 포함 여부
- `missing_bbox_counts` / `overlap_bbox_counts`: 클래스 번호·개수 요약
- `issue_image_lookup`: 이미지 번호에 해당하는 파일명·경로

표가 길면 앞 50행만 표시하며 전체 결과는 위 변수에 남습니다. `ISSUE_PREVIEW_ROWS`로 표시 행 수를 조절합니다. 원본·보정 기록은 수정하지 않습니다.

`additional_image_issues`에 문제가 있는 이미지별 누락·겹침 개수를 모읍니다. 마지막에 유형별 최대 3장의 원본과 기존 bbox를 표시합니다. 누락 코드는 제목, 겹침 박스는 빨간색으로 표시합니다. 파일명에 없는 알약의 누락은 이 검사만으로 찾을 수 없습니다.


In [ ]:
from data_process.data_processing import find_box_pairs

ISSUE_PREVIEW_ROWS = 50
if any(name not in globals() for name in ["selected_pills", "image_index", "available_ann_df"]):
    raise RuntimeError("먼저 7절 후보 선별까지 실행하세요.")

issue_files = sorted(set(selected_pills["file_name"]))
issue_image_lookup = (
    image_index[image_index["file_name"].isin(issue_files)][["file_name", "path"]]
    .sort_values("file_name")
    .reset_index(drop=True)
)
issue_image_lookup.insert(0, "이미지 번호", np.arange(1, len(issue_image_lookup) + 1))
issue_image_numbers = issue_image_lookup.set_index("file_name")["이미지 번호"]
issue_annotations = available_ann_df[available_ann_df["file_name"].isin(issue_files)].copy()

# 이미지에 함께 찍힌 모든 약을 확인한다. 보충 대상 클래스만으로 제한하지 않는다.
expected_image_pills = issue_image_lookup[["file_name"]].copy()
expected_image_pills["category_id"] = expected_image_pills["file_name"].map(
    lambda name: sorted({int(code) for code in name.split("_")[0].split("-")[1:]})
)
expected_image_pills = expected_image_pills.explode("category_id")
expected_image_pills["category_id"] = expected_image_pills["category_id"].astype(int)
existing_pairs = issue_annotations[["file_name", "category_id"]].drop_duplicates()
image_pill_presence = expected_image_pills.merge(
    existing_pairs,
    on=["file_name", "category_id"],
    how="left",
    indicator=True,
    validate="one_to_one",
)
missing_labels = image_pill_presence[image_pill_presence["_merge"].eq("left_only")][
    ["file_name", "category_id"]
].copy()
missing_labels["유형"] = "bbox 라벨 없음"
coordinate_values = issue_annotations[["x", "y", "w", "h"]].apply(pd.to_numeric, errors="coerce")
finite_coordinates = np.isfinite(coordinate_values).all(axis=1)
missing_coordinates = (
    issue_annotations.loc[~finite_coordinates, ["file_name", "category_id"]]
    .drop_duplicates()
    .copy()
)
missing_coordinates["유형"] = "bbox 좌표 결측·비정상값"
missing_bbox_cases = pd.concat([missing_labels, missing_coordinates], ignore_index=True)
missing_bbox_cases["이미지 번호"] = missing_bbox_cases["file_name"].map(issue_image_numbers)
missing_bbox_cases = (
    missing_bbox_cases.rename(columns={"category_id": "클래스 번호"})[
        ["이미지 번호", "클래스 번호", "유형", "file_name"]
    ]
    .sort_values(["클래스 번호", "이미지 번호"])
    .reset_index(drop=True)
)
missing_bbox_counts = missing_bbox_cases.groupby("클래스 번호").size().rename("개수").reset_index()

# 양의 면적을 가진 유한 좌표만 겹침 계산에 사용한다. 결측은 위 표에 따로 남긴다.
overlap_input = issue_annotations[
    finite_coordinates & coordinate_values[["w", "h"]].gt(0).all(axis=1)
].copy()
pairs = find_box_pairs(overlap_input, max_gap=0)
overlap_bbox_cases = pairs[pairs["iou"].gt(0)].copy()
selected_label_keys = set(
    zip(selected_pills["source_json"], selected_pills["annotation_id"], strict=True)
)
overlap_bbox_cases["선택 bbox 포함"] = [
    (row.source_json_1, row.annotation_id_1) in selected_label_keys
    or (row.source_json_2, row.annotation_id_2) in selected_label_keys
    for row in overlap_bbox_cases.itertuples()
]
overlap_bbox_cases["이미지 번호"] = overlap_bbox_cases["file_name"].map(issue_image_numbers)
overlap_bbox_cases = (
    overlap_bbox_cases.rename(
        columns={"category_id_1": "클래스 1", "category_id_2": "클래스 2", "iou": "IoU"}
    )
    .sort_values(["IoU", "이미지 번호"], ascending=[False, True])
    .reset_index(drop=True)
)
# 한 클래스가 여러 bbox 쌍에 겹치면 쌍별로 센다. 같은 클래스끼리의 쌍은 한 번 센다.
overlap_class_rows = pd.concat(
    [
        overlap_bbox_cases[["클래스 1"]].rename(columns={"클래스 1": "클래스 번호"}),
        overlap_bbox_cases.loc[
            overlap_bbox_cases["클래스 1"].ne(overlap_bbox_cases["클래스 2"]), ["클래스 2"]
        ].rename(columns={"클래스 2": "클래스 번호"}),
    ],
    ignore_index=True,
)
overlap_bbox_counts = overlap_class_rows.groupby("클래스 번호").size().rename("개수").reset_index()
print(f"검사: 추가 예정 이미지 {len(issue_image_lookup):,}장")
missing_image_count = missing_bbox_cases["file_name"].nunique()
print(f"이미지 내 전체 알약 bbox 누락: {len(missing_bbox_cases):,}건 / {missing_image_count:,}장")
print(f"bbox 겹침: {len(overlap_bbox_cases):,}쌍 / {overlap_bbox_cases['file_name'].nunique():,}장")
print(f"그중 선택한 알약 bbox가 포함된 겹침: {overlap_bbox_cases['선택 bbox 포함'].sum():,}쌍")
print("1. bbox 누락 — 클래스 번호 / 개수")
with pd.option_context("display.max_rows", None):
    display(missing_bbox_counts.style.hide(axis="index"))
display(
    missing_bbox_cases[["이미지 번호", "클래스 번호", "유형"]]
    .head(ISSUE_PREVIEW_ROWS)
    .style.hide(axis="index")
)
print("2. bbox 겹침 — 클래스 번호 / 관여한 겹침 쌍 개수")
with pd.option_context("display.max_rows", None):
    display(overlap_bbox_counts.style.hide(axis="index"))
display(
    overlap_bbox_cases[["이미지 번호", "클래스 1", "클래스 2", "IoU", "선택 bbox 포함"]]
    .head(ISSUE_PREVIEW_ROWS)
    .style.hide(axis="index")
    .format({"IoU": "{:.4f}"})
)
# 문제가 있는 원본 이미지 목록을 한곳에 모은다.
additional_image_issues = issue_image_lookup.copy()
additional_image_issues["누락"] = (
    additional_image_issues["file_name"]
    .map(missing_bbox_cases.groupby("file_name").size())
    .fillna(0)
    .astype(int)
)
additional_image_issues["겹침"] = (
    additional_image_issues["file_name"]
    .map(overlap_bbox_cases.groupby("file_name").size())
    .fillna(0)
    .astype(int)
)
additional_image_issues = additional_image_issues[
    additional_image_issues["누락"].gt(0) | additional_image_issues["겹침"].gt(0)
].reset_index(drop=True)
print(f"결측·겹침 점검 대상 원본 이미지: {len(additional_image_issues):,}장")
display(
    additional_image_issues[["이미지 번호", "누락", "겹침"]]
    .head(ISSUE_PREVIEW_ROWS)
    .style.hide(axis="index")
)

ISSUE_IMAGE_EXAMPLES = 3
for issue_type, cases in [("누락", missing_bbox_cases), ("겹침", overlap_bbox_cases)]:
    example_files = cases["file_name"].drop_duplicates().head(ISSUE_IMAGE_EXAMPLES)
    if example_files.empty:
        print(f"{issue_type}: 해당 이미지 없음")
        continue
    fig, axes = image_grid(len(example_files), cols=3)
    for ax, name in zip(axes, example_files, strict=True):
        original_boxes = issue_annotations[issue_annotations["file_name"].eq(name)]
        if issue_type == "누락":
            codes = missing_bbox_cases.loc[
                missing_bbox_cases["file_name"].eq(name), "클래스 번호"
            ].tolist()
            detail = f"bbox 없는 코드: {codes}"
        else:
            overlaps = overlap_bbox_cases[overlap_bbox_cases["file_name"].eq(name)]
            codes = sorted(set(overlaps["클래스 1"]) | set(overlaps["클래스 2"]))
            detail = f"겹친 코드: {codes}"
        show_image(
            ax,
            img_path[name],
            title=f"이미지 {issue_image_numbers[name]} / {issue_type}\n{detail}",
            fontsize=8,
        )
        draw_bboxes(ax, original_boxes, label_col="category_id", color="gray")
        if issue_type == "겹침":
            draw_bboxes(
                ax,
                original_boxes[original_boxes["category_id"].isin(codes)],
                label_col="category_id",
                color="red",
            )
    fig.tight_layout()
    plt.show()

## 13. 대회 클래스만 들어 있는 외부 이미지와 분포 변화

**개별 알약 crop 후보가 아닌 원본 이미지 전체**를 기준으로 봅니다.

1. 파일명에 있는 모든 약 코드가 대회 클래스인 외부 이미지 수를 셉니다. 파일명 기준이므로 실제 이미지 속 알약 종류는 검수가 필요합니다.
2. 그중 라벨의 클래스도 대회 클래스이고, 파일명과 라벨이 일치하며, bbox 좌표·중복 검사를 통과한 이미지를 분포 계산에 사용합니다. 대회와 동일 파일명은 중복 추가하지 않습니다.
3. 해당 이미지의 **모든 bbox를 한 번씩** 대회 보정 v1에 더한 예상 분포를 표시합니다. 클래스당 157개로 제한하지 않습니다.

동일 combo라는 이유만으로 수량 집계에서 빼지 않습니다. 기존 6절의 212장은 동일 combo도 제외한 더 엄격한 후보이므로 집계 기준이 다릅니다.
이 셀은 전체 데이터의 분포 비교입니다. 실제 train 추가 시에는 fold별 validation combo·촬영 세팅과의 중복, 이미지 내용 중복, bbox 겹침을 별도 검수해야 합니다. **bbox 겹침은 이 수량에서 별도로 제외하지 않았습니다.** 데이터는 병합하거나 수정하지 않습니다.

- `competition_class_only_images`: 파일명 기준 대회 클래스만 있는 전체 이미지
- `class_only_add_images` / `class_only_add_annotations`: 분포 계산에 사용한 이미지와 전체 라벨
- `class_only_balance_counts`: 클래스 번호·대회 현재·외부 추가·추가 후
- `class_only_imbalance_metrics`: 추가 전후 불균형 지표. CV·최대/최소 비율·Gini가 낮을수록 균등합니다.


In [ ]:
if any(
    name not in globals()
    for name in ["comp_corrected_df", "available_ann_df", "img_df", "candidate_image_review"]
):
    raise RuntimeError("먼저 6절의 외부 이미지 후보 검사까지 실행하세요.")

competition_class_ids = set(comp_corrected_df["category_id"].astype(int))
competition_class_only_review = img_df[["file_name", "combo", "path"]].copy()
competition_class_only_review["filename_classes"] = competition_class_only_review["combo"].map(
    lambda combo: {int(code) for code in combo.split("-")[1:]}
)
competition_class_only_review["only_competition_classes"] = competition_class_only_review[
    "filename_classes"
].map(lambda codes: bool(codes) and codes.issubset(competition_class_ids))
competition_class_only_images = competition_class_only_review[
    competition_class_only_review["only_competition_classes"]
].copy()
class_only_checks = candidate_image_review.set_index("file_name")
label_classes_by_file = available_ann_df.groupby("file_name")["category_id"].agg(set)
competition_class_only_images["label_classes_match"] = competition_class_only_images[
    "file_name"
].map(
    lambda name: (
        bool(label_classes_by_file.get(name, set()))
        and label_classes_by_file[name].issubset(competition_class_ids)
    )
)
for column in ["complete_codes", "valid_boxes"]:
    competition_class_only_images[column] = competition_class_only_images["file_name"].map(
        class_only_checks[column]
    )
competition_class_only_images["same_competition_filename"] = competition_class_only_images[
    "file_name"
].isin(comp_image_names)
class_only_add_images = competition_class_only_images[
    competition_class_only_images["label_classes_match"]
    & competition_class_only_images["complete_codes"]
    & competition_class_only_images["valid_boxes"]
    & ~competition_class_only_images["same_competition_filename"]
].copy()
class_only_add_annotations = available_ann_df[
    available_ann_df["file_name"].isin(class_only_add_images["file_name"])
].copy()
assert set(class_only_add_annotations["category_id"]).issubset(competition_class_ids)

print("대회 클래스만 들어 있는 외부 이미지 (파일명 기준)")
display(
    pd.DataFrame(
        [
            ["전체 외부 이미지", len(img_df)],
            ["대회 클래스만 있는 이미지", len(competition_class_only_images)],
            ["라벨·좌표 검사 후 분포에 사용할 이미지", len(class_only_add_images)],
            ["추가되는 전체 bbox", len(class_only_add_annotations)],
            ["참고: 동일 combo까지 제외한 기존 후보", len(candidate_images)],
        ],
        columns=["구분", "개수"],
    ).style.hide(axis="index")
)

class_only_balance_counts = (
    comp_corrected_df.groupby("category_id")
    .size()
    .rename_axis("클래스 번호")
    .rename("대회 현재")
    .to_frame()
)
class_only_balance_counts["외부 추가"] = (
    class_only_add_annotations.groupby("category_id")
    .size()
    .reindex(class_only_balance_counts.index, fill_value=0)
)
class_only_balance_counts["추가 후"] = (
    class_only_balance_counts["대회 현재"] + class_only_balance_counts["외부 추가"]
)
print("클래스 번호별 bbox 개수 — 157개 제한 없이 이미지의 모든 라벨 추가")
with pd.option_context("display.max_rows", None):
    display(class_only_balance_counts.reset_index().style.hide(axis="index"))

metric_rows = []
common_scope_ids = sorted(competition_class_ids & set(available_ann_df["category_id"]))
for scope, scope_ids in [
    ("대회 전체 클래스", class_only_balance_counts.index),
    ("공통 클래스만", common_scope_ids),
]:
    for column, stage in [("대회 현재", "추가 전"), ("추가 후", "추가 후")]:
        values = class_only_balance_counts.loc[scope_ids, column].to_numpy(dtype=float)
        sorted_values = np.sort(values)
        total = values.sum()
        gini = (
            (
                2 * np.dot(np.arange(1, len(values) + 1), sorted_values) / (len(values) * total)
                - (len(values) + 1) / len(values)
            )
            if total > 0
            else np.nan
        )
        metric_rows.append(
            {
                "범위": scope,
                "시점": stage,
                "최소 bbox": int(values.min()),
                "최대 bbox": int(values.max()),
                "최대/최소": values.max() / values.min() if values.min() > 0 else np.inf,
                "CV": values.std(ddof=0) / values.mean() if values.mean() > 0 else np.nan,
                "Gini": max(0.0, gini) if np.isfinite(gini) else np.nan,
            }
        )
class_only_imbalance_metrics = pd.DataFrame(metric_rows)
print("불균형 비교 — 최대/최소·CV·Gini가 낮을수록 균등")
display(
    class_only_imbalance_metrics.style.hide(axis="index").format(
        {"최대/최소": "{:.2f}", "CV": "{:.3f}", "Gini": "{:.3f}"}
    )
)

# 클래스 번호를 고정된 순서로 놓아 전후 분포를 직접 비교한다.
fig, axes = plt.subplots(1, 2, figsize=(17, 11), sharex=True)
for ax, positions in zip(
    axes, np.array_split(np.arange(len(class_only_balance_counts)), 2), strict=True
):
    part = class_only_balance_counts.iloc[positions]
    labels = part.index.astype(str)
    ax.barh(labels, part["대회 현재"], label="대회 현재", color="steelblue")
    ax.barh(
        labels, part["외부 추가"], left=part["대회 현재"], label="외부 추가", color="darkorange"
    )
    ax.bar_label(ax.containers[1], labels=part["추가 후"].astype(str), padding=3, fontsize=8)
    ax.invert_yaxis()
    ax.set(xlabel="bbox 개수", ylabel="클래스 번호")
    ax.set_xlim(0, max(1, class_only_balance_counts["추가 후"].max()) * 1.18)
    ax.legend()
fig.suptitle("대회 클래스만 있는 외부 이미지를 모두 추가했을 때의 클래스 분포")
fig.tight_layout()
plt.show()

# 전체 클래스의 비율 변화와 불균형 지표를 같은 화면에 비교한다.
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
ratios = (
    class_only_balance_counts[["대회 현재", "추가 후"]].div(
        class_only_balance_counts[["대회 현재", "추가 후"]].sum(), axis=1
    )
    * 100
)
axes[0].plot(ratios.index.astype(str), ratios["대회 현재"], label="추가 전", marker=".")
axes[0].plot(ratios.index.astype(str), ratios["추가 후"], label="추가 후", marker=".")
axes[0].set(title="클래스 비율 변화", xlabel="클래스 번호", ylabel="전체 bbox 중 비율 (%)")
axes[0].tick_params(axis="x", labelrotation=90, labelsize=6)
axes[0].legend()
for ax, metric in zip(axes[1:], ["CV", "Gini"], strict=True):
    metric_chart = class_only_imbalance_metrics.pivot(index="범위", columns="시점", values=metric)
    metric_chart = metric_chart[["추가 전", "추가 후"]]
    metric_chart.plot.bar(ax=ax, rot=0, color=["steelblue", "darkorange"])
    for container in ax.containers:
        ax.bar_label(container, fmt="%.3f", padding=3, fontsize=8)
    ax.set(title=f"{metric} 비교 (낮을수록 균등)", xlabel="", ylabel=metric)
    ax.margins(y=0.2)
fig.tight_layout()
plt.show()

## 14. 추가 예정 이미지에 함께 들어오는 외부 전용 클래스

검사 범위는 **7절의 `selected_pills`가 들어 있는 원본 이미지 전체**입니다. 개별 bbox만 crop하는 경우가 아니라, 이 원본 이미지들을 통째로 대회 데이터에 추가한다고 가정합니다. 13절의 대회 클래스만 있는 212장과는 다른 범위입니다.

- 같은 이미지는 한 번만 세고 그 안의 **모든 라벨**을 집계합니다. 선택하지 않은 공통 알약과 외부 전용 알약도 포함합니다.
- 외부 전용 클래스 = 대회 보정 v1의 클래스 집합에 없는 약 코드입니다.
- `accompanying_foreign_distribution`: 클래스 번호·bbox 개수·bbox 있는 이미지 수·파일명 기준 이미지 수·bbox 없는 이미지 수
- `whole_image_augmented_counts`: 원본 이미지를 통째로 추가했을 때 모든 클래스의 대회 현재·외부 추가·추가 후 분포
- `accompanying_image_lookup`: 집계한 원본 이미지의 경로

**bbox 개수는 실제 존재하는 라벨만 셉니다.** 파일명에는 있지만 bbox가 없는 외부 약은 별도 열에 표시합니다. 파일명 기준 약 종류는 시각 검수가 필요합니다. 원본 이미지에 외부 약이 보이는데 해당 라벨만 지우는 방식으로 학습하면 미라벨 객체가 남습니다. 이 셀은 개수와 분포만 확인하며 학습 데이터나 클래스 매핑을 바꾸지 않습니다.


In [ ]:
if any(
    name not in globals()
    for name in ["selected_pills", "comp_corrected_df", "available_ann_df", "image_index"]
):
    raise RuntimeError("먼저 7절의 공통 알약 선별까지 실행하세요.")

accompanying_names = set(selected_pills["file_name"])
accompanying_image_lookup = (
    image_index[image_index["file_name"].isin(accompanying_names)][["file_name", "path"]]
    .sort_values("file_name")
    .reset_index(drop=True)
)
accompanying_annotations = available_ann_df[
    available_ann_df["file_name"].isin(accompanying_names)
].copy()
base_class_ids = set(comp_corrected_df["category_id"].astype(int))
accompanying_foreign_annotations = accompanying_annotations[
    ~accompanying_annotations["category_id"].isin(base_class_ids)
].copy()

# 이미지 속 외부 약이 라벨 집계에서 사라지지 않도록 파일명 약 코드도 별도로 대조한다.
accompanying_expected = accompanying_image_lookup[["file_name"]].copy()
accompanying_expected["category_id"] = accompanying_expected["file_name"].map(
    lambda name: sorted({int(code) for code in name.split("_")[0].split("-")[1:]})
)
accompanying_expected = accompanying_expected.explode("category_id")
accompanying_expected["category_id"] = accompanying_expected["category_id"].astype(int)
foreign_expected = accompanying_expected[
    ~accompanying_expected["category_id"].isin(base_class_ids)
].copy()
foreign_presence = foreign_expected.merge(
    accompanying_annotations[["file_name", "category_id"]].drop_duplicates(),
    on=["file_name", "category_id"],
    how="left",
    indicator=True,
    validate="one_to_one",
)
foreign_missing = foreign_presence[foreign_presence["_merge"].eq("left_only")]
foreign_class_ids = sorted(
    set(foreign_expected["category_id"]) | set(accompanying_foreign_annotations["category_id"])
)
accompanying_foreign_distribution = pd.DataFrame(
    index=pd.Index(foreign_class_ids, name="클래스 번호")
)
accompanying_foreign_distribution["bbox 개수"] = (
    accompanying_foreign_annotations.groupby("category_id")
    .size()
    .reindex(foreign_class_ids, fill_value=0)
)
accompanying_foreign_distribution["bbox 있는 이미지"] = (
    accompanying_foreign_annotations.groupby("category_id")["file_name"]
    .nunique()
    .reindex(foreign_class_ids, fill_value=0)
)
accompanying_foreign_distribution["파일명 기준 이미지"] = (
    foreign_expected.groupby("category_id")["file_name"]
    .nunique()
    .reindex(foreign_class_ids, fill_value=0)
)
accompanying_foreign_distribution["bbox 없는 이미지"] = (
    foreign_missing.groupby("category_id")["file_name"]
    .nunique()
    .reindex(foreign_class_ids, fill_value=0)
)

n_with_foreign = foreign_expected["file_name"].nunique()
n_foreign_label_classes = accompanying_foreign_annotations["category_id"].nunique()
n_foreign_bbox = len(accompanying_foreign_annotations)
n_all_bbox = len(accompanying_annotations)
foreign_bbox_ratio = n_foreign_bbox / n_all_bbox * 100 if n_all_bbox else 0.0
print(f"원본 이미지 전체 추가 가정: {len(accompanying_image_lookup):,}장")
print(f"외부 전용 약이 함께 있는 이미지: {n_with_foreign:,}장 (파일명 기준)")
print(
    f"함께 들어오는 외부 전용 클래스: {len(foreign_class_ids)}종 / "
    f"라벨이 있는 클래스: {n_foreign_label_classes}종"
)
print(f"외부 전용 bbox: {n_foreign_bbox:,}개 / 추가 이미지 전체 bbox의 {foreign_bbox_ratio:.1f}%")
print(f"외부 전용 약 중 bbox 누락: {len(foreign_missing):,}건")
print("외부 전용 클래스별 개수와 이미지 수")
with pd.option_context("display.max_rows", None):
    display(accompanying_foreign_distribution.reset_index().style.hide(axis="index"))

if not accompanying_foreign_distribution.empty:
    fig, axes = plt.subplots(1, 2, figsize=(16, max(6, len(foreign_class_ids) * 0.22)))
    for ax, positions in zip(
        axes, np.array_split(np.arange(len(foreign_class_ids)), 2), strict=True
    ):
        part = accompanying_foreign_distribution.iloc[positions]
        ax.barh(part.index.astype(str), part["bbox 개수"], color="darkorange")
        ax.bar_label(ax.containers[0], padding=3, fontsize=8)
        ax.invert_yaxis()
        ax.set(xlabel="bbox 개수", ylabel="대회에 없는 클래스 번호")
        max_count = max(1, accompanying_foreign_distribution["bbox 개수"].max())
        ax.set_xlim(0, max_count * 1.2)
    fig.suptitle("추가 예정 원본 이미지에 함께 들어오는 외부 전용 클래스 분포")
    fig.tight_layout()
    plt.show()
else:
    print("함께 들어오는 외부 전용 클래스가 없습니다.")

# 통째로 가져올 경우, 선택한 알약뿐 아니라 같은 이미지의 모든 bbox가 추가된다.
all_carried_classes = sorted(base_class_ids | set(accompanying_annotations["category_id"]))
whole_image_augmented_counts = pd.DataFrame(index=pd.Index(all_carried_classes, name="클래스 번호"))
whole_image_augmented_counts["대회 현재"] = (
    comp_corrected_df.groupby("category_id").size().reindex(all_carried_classes, fill_value=0)
)
whole_image_augmented_counts["외부 추가"] = (
    accompanying_annotations.groupby("category_id")
    .size()
    .reindex(all_carried_classes, fill_value=0)
)
whole_image_augmented_counts["추가 후"] = (
    whole_image_augmented_counts["대회 현재"] + whole_image_augmented_counts["외부 추가"]
)
whole_image_augmented_counts["외부 전용"] = ~whole_image_augmented_counts.index.isin(base_class_ids)
print("원본 이미지 전체를 추가할 때의 전체 클래스 분포")
with pd.option_context("display.max_rows", None):
    display(whole_image_augmented_counts.reset_index().style.hide(axis="index"))

fig, axes = plt.subplots(1, 2, figsize=(18, 5))
labels = whole_image_augmented_counts.index.astype(str)
bar_colors = np.where(whole_image_augmented_counts["외부 전용"], "darkorange", "steelblue")
axes[0].bar(labels, whole_image_augmented_counts["추가 후"], color=bar_colors)
axes[0].tick_params(axis="x", labelrotation=90, labelsize=6)
axes[0].set(
    title="원본 이미지 전체 추가 후 분포 (주황: 외부 전용)",
    xlabel="클래스 번호",
    ylabel="bbox 개수",
)
composition_counts = [n_all_bbox - n_foreign_bbox, n_foreign_bbox]
axes[1].bar(
    ["대회에 있는 클래스", "대회에 없는 클래스"],
    composition_counts,
    color=["steelblue", "darkorange"],
)
axes[1].bar_label(axes[1].containers[0], padding=3)
axes[1].margins(y=0.15)
axes[1].set(title="추가 이미지에 포함된 전체 bbox 구성", ylabel="bbox 개수")
fig.tight_layout()
plt.show()

## 15. 개수 상한 없이 불균형 조절 — 외부 전용 약 0~1개 이미지

**최종 선별 기준입니다.** 앞 절의 157개 기준 실험과 별도로, 여기서는 목표 개수와 상한을 모두 없앴습니다. 실제 학습·병합은 실행하지 않습니다.

- 원본 이미지에 대회 클래스 알약이 있고, **외부 전용 알약은 0개 또는 1개**여야 합니다. 파일명과 모든 라벨을 대조합니다.
- bbox 누락·좌표 오류·중복 라벨·대회와 동일 파일명은 제외합니다. 동일 combo는 수량 분석에 포함하며 실제 학습 시 validation 누수를 확인해야 합니다.
- 이미지 안의 모든 대회 클래스 bbox를 계산하고 **전체 대회 클래스의 CV(표준편차 ÷ 평균)**를 줄입니다. 외부 데이터에 없는 2개 클래스도 포함합니다.
- 클래스별 수는 157개보다 많거나 적을 수 있습니다. 연속 최적화를 여러 초기값에서 수행한 후 정수 이미지 수로 바꾸고, 그룹별 수량 조정과 두 그룹 교환으로 CV를 더 줄입니다. 전역 최적해를 보장하는 방식은 아닙니다.
- 확보한 클래스 분포를 유지하면서 외부 전용 bbox 수, 이미지 수를 차례로 최소화합니다. 이 정수 최적화는 단계별 최대 15초이며 성공 여부를 표시합니다.

**최종 결과 변수:** `flexible_selected_images`, `flexible_selected_annotations`, `flexible_balance_report`, `flexible_imbalance_metrics`.
겹침은 자동으로 오류라고 단정해 제외하지 않으며, 선별 후 겹침 쌍 수를 따로 표시합니다. 원본 이미지는 이동·수정하지 않습니다.


In [ ]:
from scipy.optimize import Bounds, LinearConstraint, milp, minimize
from scipy.sparse import csc_array
from threadpoolctl import threadpool_limits

from data_process.data_processing import find_box_pairs

if any(
    name not in globals()
    for name in ["comp_corrected_df", "available_ann_df", "candidate_image_review", "class_balance"]
):
    raise RuntimeError("먼저 6절의 후보 검사까지 실행하세요.")

MAX_FOREIGN_PER_IMAGE = 1
SELECTION_TIME_LIMIT = 15.0
flexible_base_counts = comp_corrected_df.groupby("category_id").size().sort_index()
flexible_competition_ids = set(flexible_base_counts.index)
flexible_common_ids = sorted(flexible_competition_ids & set(available_ann_df["category_id"]))
flexible_image_review = candidate_image_review.copy()
flexible_image_review["name_codes"] = flexible_image_review["combo"].map(
    lambda combo: {int(code) for code in combo.split("-")[1:]}
)
flexible_image_review["foreign_count"] = flexible_image_review["name_codes"].map(
    lambda codes: len(codes - flexible_competition_ids)
)
flexible_image_review["competition_count"] = flexible_image_review["name_codes"].map(
    lambda codes: len(codes & flexible_competition_ids)
)
flexible_composition_mask = flexible_image_review["competition_count"].gt(
    0
) & flexible_image_review["foreign_count"].le(MAX_FOREIGN_PER_IMAGE)
flexible_candidates = (
    flexible_image_review[
        flexible_composition_mask
        & flexible_image_review["complete_codes"]
        & flexible_image_review["valid_boxes"]
        & ~flexible_image_review["file_name"].isin(comp_image_names)
    ]
    .sort_values("file_name")
    .copy()
)
flexible_candidate_annotations = available_ann_df[
    available_ann_df["file_name"].isin(flexible_candidates["file_name"])
].copy()
print("조건별 이미지 수")
flexible_candidate_summary = pd.DataFrame(
    [
        {
            "외부 전용 수": count,
            "파일명 기준": int(
                (flexible_composition_mask & flexible_image_review["foreign_count"].eq(count)).sum()
            ),
            "라벨 검사 후": int(flexible_candidates["foreign_count"].eq(count).sum()),
        }
        for count in [0, 1]
    ]
)
display(flexible_candidate_summary.style.hide(axis="index"))

# 정해진 목표 수 없이 전체 대회 클래스의 CV를 줄인다.
flexible_ids = flexible_base_counts.index.tolist()
flexible_matrix = pd.crosstab(
    flexible_candidate_annotations["file_name"], flexible_candidate_annotations["category_id"]
).reindex(index=flexible_candidates["file_name"], columns=flexible_ids, fill_value=0)
flexible_foreign_by_name = flexible_candidates.set_index("file_name")["foreign_count"]
flexible_signature_files = {}
for name, row in flexible_matrix.iterrows():
    signature = (*row.to_numpy(dtype=int).tolist(), int(flexible_foreign_by_name[name]))
    flexible_signature_files.setdefault(signature, []).append(name)
if not flexible_signature_files:
    raise RuntimeError("조건에 맞는 외부 이미지가 없습니다.")
flexible_signatures = np.asarray(list(flexible_signature_files), dtype=float)
flexible_group_counts = flexible_signatures[:, :-1].T
flexible_foreign_cost = flexible_signatures[:, -1]
flexible_capacities = np.asarray(
    [len(files) for files in flexible_signature_files.values()], dtype=int
)
flexible_base = flexible_base_counts.to_numpy(dtype=float)
n_classes, n_groups = flexible_group_counts.shape
flexible_scaled_counts = flexible_group_counts * flexible_capacities[None, :]
flexible_scaled_totals = flexible_scaled_counts.sum(axis=0)


def flexible_cv_squared(fractions: np.ndarray) -> float:
    """클래스별 목표 수 없이, 선택 비율에 따른 CV 제곱을 계산한다."""
    counts = flexible_base + flexible_scaled_counts @ fractions
    return float(n_classes * (counts @ counts) / counts.sum() ** 2 - 1)


def flexible_cv_gradient(fractions: np.ndarray) -> np.ndarray:
    """선택 비율별 CV 제곱의 해석적 기울기를 계산한다."""
    counts = flexible_base + flexible_scaled_counts @ fractions
    total = counts.sum()
    return (
        2
        * n_classes
        / total**2
        * (flexible_scaled_counts.T @ counts - (counts @ counts) / total * flexible_scaled_totals)
    )


# 여러 초기값에서 연속 최적화를 하고 실제 정수 이미지 수로 바꿔 비교한다.
flexible_best_quantities = np.zeros(n_groups, dtype=int)
flexible_best_score = flexible_cv_squared(np.zeros(n_groups))
flexible_solver_log = []
# 작은 수치 최적화에서 BLAS 스레드 과다 사용을 피한다. 학습 설정에는 영향을 주지 않는다.
with threadpool_limits(limits=1, user_api="blas"):
    for initial_fraction in [0.0, 0.5, 1.0]:
        result = minimize(
            flexible_cv_squared,
            np.full(n_groups, initial_fraction),
            jac=flexible_cv_gradient,
            method="SLSQP",
            bounds=Bounds(np.zeros(n_groups), np.ones(n_groups)),
            options={"maxiter": 1000, "ftol": 1e-12},
        )
        flexible_solver_log.append(
            {
                "단계": f"CV 연속 최적화 ({initial_fraction})",
                "수렴·해 확인": bool(result.success),
            }
        )
        quantities = np.rint(np.clip(result.x, 0, 1) * flexible_capacities).astype(int)
        candidate_counts = flexible_base + flexible_group_counts @ quantities
        candidate_score = (
            n_classes * (candidate_counts @ candidate_counts) / candidate_counts.sum() ** 2 - 1
        )
        if candidate_score < flexible_best_score:
            flexible_best_score = candidate_score
            flexible_best_quantities = quantities.copy()

    # 각 그룹의 정수 선택 수를 한 번에 바꾸고, 2개 그룹의 ±1 교환도 비교해 CV를 더 줄인다.
    flexible_integer_steps = 0
    for _adjustment in range(200):
        counts = flexible_base + flexible_group_counts @ flexible_best_quantities
        total = counts.sum()
        square_sum = counts @ counts
        group_totals = flexible_group_counts.sum(axis=0)
        group_squares = (flexible_group_counts**2).sum(axis=0)
        group_dots = flexible_group_counts.T @ counts
        root_denominator = group_squares * total - group_dots * group_totals
        roots = np.divide(
            group_totals * square_sum - group_dots * total,
            root_denominator,
            out=np.zeros(n_groups),
            where=np.abs(root_denominator) > 1e-12,
        )
        lower_steps = -flexible_best_quantities
        upper_steps = flexible_capacities - flexible_best_quantities
        step_options = np.stack(
            [
                lower_steps,
                upper_steps,
                np.clip(np.floor(roots), lower_steps, upper_steps),
                np.clip(np.ceil(roots), lower_steps, upper_steps),
            ]
        ).astype(int)
        changed_total = total + step_options * group_totals
        changed_square_sum = (
            square_sum + 2 * step_options * group_dots + step_options**2 * group_squares
        )
        scores = n_classes * changed_square_sum / changed_total**2 - 1
        row, column = np.unravel_index(scores.argmin(), scores.shape)
        if scores[row, column] < flexible_best_score - 1e-12:
            flexible_best_quantities[column] += step_options[row, column]
            flexible_best_score = float(scores[row, column])
            flexible_integer_steps += 1
            continue
        # 반올림 때문에 한 그룹만 바꿔서는 개선되지 않을 때 두 그룹을 동시에 조정한다.
        group_indices = np.tile(np.arange(n_groups), 2)
        move_signs = np.concatenate([np.ones(n_groups, dtype=int), -np.ones(n_groups, dtype=int)])
        valid_moves = np.concatenate(
            [
                flexible_best_quantities < flexible_capacities,
                flexible_best_quantities > 0,
            ]
        )
        move_vectors = np.concatenate([flexible_group_counts.T, -flexible_group_counts.T])
        move_totals = move_vectors.sum(axis=1)
        move_dots = move_vectors @ counts
        move_squares = (move_vectors**2).sum(axis=1)
        pair_total = total + move_totals[:, None] + move_totals[None, :]
        pair_square = (
            square_sum
            + 2 * (move_dots[:, None] + move_dots[None, :])
            + move_squares[:, None]
            + move_squares[None, :]
            + 2 * move_vectors @ move_vectors.T
        )
        pair_scores = n_classes * pair_square / pair_total**2 - 1
        allowed_pairs = (
            valid_moves[:, None]
            & valid_moves[None, :]
            & (group_indices[:, None] != group_indices[None, :])
        )
        pair_scores[~allowed_pairs] = np.inf
        first, second = np.unravel_index(pair_scores.argmin(), pair_scores.shape)
        if pair_scores[first, second] >= flexible_best_score - 1e-12:
            break
        flexible_best_quantities[group_indices[first]] += move_signs[first]
        flexible_best_quantities[group_indices[second]] += move_signs[second]
        flexible_best_score = float(pair_scores[first, second])
        flexible_integer_steps += 1

# 확보한 클래스 분포를 유지하면서 외부 전용 약과 이미지 수를 차례로 줄인다.
fixed_additions = flexible_group_counts @ flexible_best_quantities
flexible_constraints = [
    LinearConstraint(csc_array(flexible_group_counts), fixed_additions, fixed_additions)
]
for cost, stage in [
    (flexible_foreign_cost, "같은 분포에서 외부 전용 약 최소화"),
    (np.ones(n_groups), "같은 분포에서 이미지 수 최소화"),
]:
    result = milp(
        cost,
        integrality=np.ones(n_groups),
        bounds=Bounds(np.zeros(n_groups), flexible_capacities),
        constraints=flexible_constraints,
        options={"time_limit": SELECTION_TIME_LIMIT, "mip_rel_gap": 0.0},
    )
    flexible_solver_log.append({"단계": stage, "수렴·해 확인": bool(result.success)})
    if result.x is None:
        print(f"{stage}: 실행 가능한 해가 없어 이전 선택을 유지합니다.")
        break
    quantities = np.rint(result.x).astype(int)
    if not np.allclose(flexible_group_counts @ quantities, fixed_additions, atol=1e-6, rtol=0):
        raise RuntimeError("선택 이미지의 클래스 분포 보존 검사를 통과하지 못했습니다.")
    flexible_best_quantities = quantities.copy()
    if stage == "같은 분포에서 외부 전용 약 최소화":
        fixed_foreign = int(flexible_foreign_cost @ quantities)
        flexible_constraints.append(
            LinearConstraint(
                flexible_foreign_cost[None, :],
                fixed_foreign,
                fixed_foreign,
            )
        )
assert (flexible_best_quantities >= 0).all()
assert (flexible_best_quantities <= flexible_capacities).all()
flexible_selected_names = []
for files, quantity in zip(
    flexible_signature_files.values(), flexible_best_quantities, strict=True
):
    flexible_selected_names.extend(files[:quantity])
flexible_selected_images = flexible_candidates[
    flexible_candidates["file_name"].isin(flexible_selected_names)
].copy()
flexible_selected_annotations = flexible_candidate_annotations[
    flexible_candidate_annotations["file_name"].isin(flexible_selected_names)
].copy()
flexible_balance_report = flexible_base_counts.rename("대회 현재").to_frame()
flexible_balance_report["추가 bbox"] = (
    flexible_selected_annotations.groupby("category_id")
    .size()
    .reindex(flexible_base_counts.index, fill_value=0)
)
flexible_balance_report["추가 후"] = (
    flexible_balance_report["대회 현재"] + flexible_balance_report["추가 bbox"]
)
flexible_balance_report.index.name = "클래스 번호"
assert flexible_selected_images["foreign_count"].le(MAX_FOREIGN_PER_IMAGE).all()
assert not flexible_selected_images["file_name"].duplicated().any()
flexible_added_foreign = int(
    (~flexible_selected_annotations["category_id"].isin(flexible_competition_ids)).sum()
)
print("157개 목표·상한 없음 / 원본 이미지 내 외부 전용 약 최대 1개")
print(f"선택 이미지 {len(flexible_selected_images):,}장")
print(f"대회 알약 추가 {flexible_balance_report['추가 bbox'].sum():,} bbox")
print(f"함께 들어오는 외부 전용 알약 {flexible_added_foreign:,} bbox")
print("CV를 줄이는 연속 최적화·정수 조정 결과이며, 전역 최적해를 보장하지 않습니다.")
with pd.option_context("display.max_rows", None):
    display(flexible_balance_report.reset_index().style.hide(axis="index"))
display(pd.DataFrame(flexible_solver_log).style.hide(axis="index"))

fig, axes = plt.subplots(1, 2, figsize=(17, 11), sharex=True)
for ax, positions in zip(
    axes, np.array_split(np.arange(len(flexible_balance_report)), 2), strict=True
):
    part = flexible_balance_report.iloc[positions]
    labels = part.index.astype(str)
    ax.barh(labels, part["대회 현재"], color="steelblue", label="대회 현재")
    ax.barh(
        labels, part["추가 bbox"], left=part["대회 현재"], color="darkorange", label="외부 추가"
    )
    ax.bar_label(ax.containers[1], labels=part["추가 후"].astype(str), padding=3, fontsize=8)
    ax.invert_yaxis()
    ax.set(xlabel="bbox 개수", ylabel="클래스 번호")
    ax.set_xlim(0, flexible_balance_report["추가 후"].max() * 1.2)
    ax.legend()
fig.suptitle("개수 상한 없이 불균형을 줄인 분포 — 외부 전용 알약 0~1개 이미지")
fig.tight_layout()
plt.show()

flexible_metric_rows = []
for scope, ids in [
    ("전체 클래스", flexible_balance_report.index),
    ("공통 클래스", flexible_common_ids),
]:
    for column, stage in [("대회 현재", "추가 전"), ("추가 후", "추가 후")]:
        values = flexible_balance_report.loc[ids, column].to_numpy(dtype=float)
        ordered = np.sort(values)
        total = values.sum()
        gini = 2 * np.dot(np.arange(1, len(values) + 1), ordered) / (len(values) * total) - (
            len(values) + 1
        ) / len(values)
        flexible_metric_rows.append(
            {
                "범위": scope,
                "시점": stage,
                "상대 평균절대편차": np.abs(values - values.mean()).sum() / total,
                "CV": values.std(ddof=0) / values.mean(),
                "Gini": max(0.0, gini),
                "최소": int(values.min()),
                "최대": int(values.max()),
            }
        )
flexible_imbalance_metrics = pd.DataFrame(flexible_metric_rows)
print("CV를 기준으로 선별했습니다. CV·Gini·상대 평균절대편차는 낮을수록 균등합니다.")
display(
    flexible_imbalance_metrics.style.hide(axis="index").format(
        {"상대 평균절대편차": "{:.3f}", "CV": "{:.3f}", "Gini": "{:.3f}"},
    )
)
flexible_pairs = find_box_pairs(flexible_selected_annotations, max_gap=0)
flexible_overlap_pairs = flexible_pairs[flexible_pairs["iou"].gt(0)].copy()
print(f"선택 이미지 내 bbox 겹침: {len(flexible_overlap_pairs):,}쌍 (검수 필요)")